# Notebook 6: Baseline Prediction & Gap Analysis

## 1. Introduction
The first step in the prescriptive process is understanding the **Current Reality**. 

Before suggesting changes, we must determine what will happen if we use the currently available resources. This notebook implements the **Baseline Prediction** logic: using the trained Phase 1 model to predict the actual production quantity based on available resources, and then classifying the result as Overproduction, Underproduction, or Balanced.

### The Workflow:
1. **Input Baseline**: Current resources (Workers, Hours, Machines) and the Planned Quantity.
2. **Predict Actual**: Use the model to find the `predicted_actual_qty`.
3. **Gap Classification**:
    - $\text{Prediction} > \text{Planned}$ $\rightarrow$ **Overproduction**
    - $\text{Prediction} < \text{Planned}$ $\rightarrow$ **Underproduction**
    - $\text{Prediction} \approx \text{Planned}$ $\rightarrow$ **Balanced**

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

# Load the saved model pipeline
MODEL_PATH = '../models/production_model.joblib'
model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")

Model loaded successfully.


## 2. Baseline Feature Builder

We need a function that can take current resources and transform them into the model's required format. Note that `planned_qty` is used for the gap analysis, but **not** as a feature for the prediction.

In [2]:
def build_baseline_features(raw_inputs):
    """
    Transforms raw resource inputs into a DataFrame for the model.
    """
    features = raw_inputs.copy()
    
    # Calculate derived features used in Phase 1
    workers = max(raw_inputs.get('no_of_workers', 1), 1)
    features['machine_worker_ratio'] = raw_inputs.get('machine_count', 0) / workers
    features['avg_hours_per_worker'] = raw_inputs.get('working_hours', 0) / workers
    
    # Standard baseline context (required by model pipeline)
    defaults = {
        'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 
        'bottleneck_flag': False, 'rate': 5266.92
    }
    for col, value in defaults.items():
        if col not in features: features[col] = value
    
    return pd.DataFrame([features])

print("Baseline feature builder defined.")

Baseline feature builder defined.


## 3. Gap Analysis Logic

This function compares the predicted actual quantity against the planned quantity and classifies the state of production.

In [3]:
def analyze_production_gap(predicted_qty, planned_qty, tolerance=2):
    """
    Classifies the production state based on the gap between prediction and plan.
    """
    gap = predicted_qty - planned_qty
    
    if abs(gap) <= tolerance:
        return 'Balanced', 'No change needed. Resources are aligned with the plan.'
    elif gap > tolerance:
        return 'Overproduction', f'Predicted output is {gap:.2f} units ABOVE plan. Suggest reducing resources.'
    else:
        return 'Underproduction', f'Predicted output is {abs(gap):.2f} units BELOW plan. Suggest increasing resources.'
    
print("Gap analysis logic defined.")

Gap analysis logic defined.


## 5. Summary\n\nThis notebook successfully implements the first stage of the prescriptive process. We can now take any current resource allocation and immediately determine if the production is on track, overproducing, or underproducing.\n\n**Validation Results:**\n- **Overproduction Detected**: Correctly identified when predicted output (66.81) exceeded plan (30).\n- **Underproduction Detected**: Correctly identified when predicted output (18.84) fell below plan (60).\n- **Balanced State Detected**: Correctly identified when prediction (35.22) was within tolerance of plan (35).\n\nThis classification now serves as the trigger for the specific optimization strategies in Notebook 8.

In [4]:
test_scenarios = [
    {
        'name': 'Scenario A: Likely Overproducing',
        'resources': {'no_of_workers': 25, 'working_hours': 1200, 'machine_count': 6, 'item_name': 'DIYANA - L - BROWN', 'factory_unit': 'Factory Unit 01', 'product_family': 'Gents', 'size_id': 10, 'grade': 'A', 'order_category': 'Others'},
        'planned_qty': 30
    },
    {
        'name': 'Scenario B: Likely Underproducing',
        'resources': {'no_of_workers': 8, 'working_hours': 300, 'machine_count': 2, 'item_name': 'DIYANA - L - BROWN', 'factory_unit': 'Factory Unit 01', 'product_family': 'Gents', 'size_id': 10, 'grade': 'A', 'order_category': 'Others'},
        'planned_qty': 60
    },
    {
        'name': 'Scenario C: Likely Balanced',
        'resources': {'no_of_workers': 15, 'working_hours': 623.96, 'machine_count': 4, 'item_name': 'DIYANA - L - BROWN', 'factory_unit': 'Factory Unit 01', 'product_family': 'Gents', 'size_id': 10, 'grade': 'A', 'order_category': 'Others'},
        'planned_qty': 35
    }
]

for scenario in test_scenarios:
    print(f"\n--- {scenario['name']} ---")
    
    # 1. Build features
    feat_df = build_baseline_features(scenario['resources'])
    
    # 2. Predict
    pred = model.predict(feat_df)[0]
    
    # 3. Analyze Gap
    status, message = analyze_production_gap(pred, scenario['planned_qty'])
    
    print(f"Planned Qty: {scenario['planned_qty']}")
    print(f"Predicted Actual Qty: {pred:.2f}")
    print(f"Status: {status}")
    print(f"Recommendation: {message}")


--- Scenario A: Likely Overproducing ---
Planned Qty: 30
Predicted Actual Qty: 66.81
Status: Overproduction
Recommendation: Predicted output is 36.81 units ABOVE plan. Suggest reducing resources.

--- Scenario B: Likely Underproducing ---
Planned Qty: 60
Predicted Actual Qty: 18.84
Status: Underproduction
Recommendation: Predicted output is 41.16 units BELOW plan. Suggest increasing resources.

--- Scenario C: Likely Balanced ---
Planned Qty: 35
Predicted Actual Qty: 35.22
Status: Balanced
Recommendation: No change needed. Resources are aligned with the plan.


## 5. Summary
This notebook successfully implements the first stage of the prescriptive process. We can now take any current resource allocation and immediately determine if the production is on track, overproducing, or underproducing. This classification will drive the specific optimization strategies in Notebook 8.